# Quasi-norm Muon: run Hypothesis 1 on Colab

**Runtime → Change runtime type → GPU** (A100 if you can get it; T4 works but is much slower).
Run top to bottom. Every sweep is **resumable**: if Colab disconnects, re-run the setup cells and the
sweep cell again and it skips finished runs (results are saved to Drive if you mount it).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## 1. Setup (results go to Drive if mounted)

In [ ]:
import os
USE_DRIVE = True   # set False to keep results only on the (ephemeral) Colab disk
RESULTS = "/content/results"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/cs182_results"
os.makedirs(RESULTS, exist_ok=True)
print("results dir:", RESULTS)

In [ ]:
%cd /content
BRANCH = "h1-colab-tuning"   # Hypothesis 1 work lives on this branch until merged into main
!git clone -b {BRANCH} https://github.com/BistroP/CS182-final-project.git repo 2>/dev/null || (cd repo && git fetch origin && git checkout {BRANCH} && git pull)
%cd /content/repo
!python -m pytest -q tests

In [ ]:
# Downloads tinyshakespeare (~1.1 MB) from github.com/karpathy/char-rnn
!python prepare_shakespeare.py

## 2. Benchmark (≈ 3–6 min) — do this first
Look at the **freon svd vs eigh** rows (`opt_step` and `s/iter`) and pick the faster one for `FREON_IMPL` below.
Please paste the output back to the team: it decides how big a sweep we can afford.

In [ ]:
!python bench.py --iters 100 --out {RESULTS}/bench.json

In [ ]:
FREON_IMPL = "eigh"   # <- set to "svd" if the benchmark shows svd is faster

## 3. Stage 1 — tune learning rates (seed 42, 1500 iters per run)
6 `(matrix_lr, base_lr)` points per optimizer, centred on the paper's Table A.1 picks (`--grid full` gives the paper's 16-point grid). 6 optimizers × 6 = 36 short runs.
Use separate `--out` files per teammate if you split `OPTS` across runtimes.


In [ ]:
OPTS = "muon kaon freon:0.6667 freon:0.75 sgd truncsgd:0.05"   # split this across teammates/runtimes if needed
AMP = "--amp"   # set to "--no-amp" if the bench showed ~0.5 s/iter for sgd (GPU without bf16, e.g. T4)
!python sweep.py tune --grid focused --opts {OPTS} --max_iters 1500 {AMP} --freon_impl {FREON_IMPL} --out {RESULTS}/tune.jsonl


In [ ]:
!python sweep.py best --out {RESULTS}/tune.jsonl

## 3b. Stage 1b — extend the SGD / TruncatedSGD lr range
Their best stage-1 points sat at the top of the grid (base_lr 2e-3; TruncatedSGD matrix_lr 1.0) with loss still improving, so the
'SGD falls behind' comparison could just reflect under-tuning. `--grid wide` adds matrix_lr {3, 10} and base_lr 4e-3 (finished points are skipped).

In [ ]:
!python sweep.py tune --grid wide --opts sgd truncsgd:0.05 --max_iters 1500 {AMP} --freon_impl {FREON_IMPL} --out {RESULTS}/tune.jsonl
!python sweep.py best --out {RESULTS}/tune.jsonl

## 4. Stage 2 — final runs, 3 seeds × 5000 iters at each optimizer's best lrs
The lrs are picked automatically from the stage-1 results (`tune*.jsonl`), so this cell can be queued right behind the tune cell.
`--rhos 1` = no rescaling (drop it for the paper's Figure-2 lr-sensitivity). Resumable like stage 1.

In [ ]:
import sys; sys.path.insert(0, "/content/repo")
import sweep
# Automatic: the best (matrix_lr, base_lr) per optimizer from stage 1 (merges every tune*.jsonl in RESULTS).
FINAL_SPECS = " ".join(sweep.best_specs(f"{RESULTS}/tune*.jsonl"))
# FINAL_SPECS = "muon:0.02:0.001 kaon:0.02:0.001 ..."   # <- or paste specs by hand instead
print("final specs:", FINAL_SPECS)
!python sweep.py scale --opts {FINAL_SPECS} --rhos 1 {AMP} --seeds 42 43 44 --freon_impl {FREON_IMPL} --out {RESULTS}/final.jsonl


## 5. Summary: mean ± std validation loss over seeds
`best` = lowest val loss seen during training (what the spec's "best validation loss" and nanoGPT's checkpointing use); `final` = loss at the last iteration.
Shakespeare-char overfits after a few thousand iterations (5000 iters ≈ 80 epochs), so `final` mostly measures overfitting; compare optimizers on `best`.

In [ ]:
import json, collections, statistics as st
rows = [json.loads(l) for l in open(f"{RESULTS}/final.jsonl")]
def ms(v): return f"{st.mean(v):.4f} ± {st.stdev(v) if len(v) > 1 else float('nan'):.4f}"
by = collections.defaultdict(list)
for r in rows:
    by[r["tag"]].append(r)
print(f"{'optimizer':16s} {'best val':18s} {'final val':18s} n")
for tag, rs in sorted(by.items(), key=lambda kv: st.mean(r["best_val_loss"] for r in kv[1])):
    print(f"{tag:16s} {ms([r['best_val_loss'] for r in rs]):18s} {ms([r['final_val_loss'] for r in rs]):18s} {len(rs)}")


## 6. Learning curves (seed 42) saved to Drive
Per-iteration curves (`log.json`) live in `runs/` on the Colab disk and vanish when the runtime resets. This links `runs/` to Drive, then re-runs seed 42 at the best lrs into a separate results file (`curves.jsonl`) so nothing in `final.jsonl` is touched. About 35 min on an L4 for the first four optimizers, ~90 min for all six.

In [ ]:
import os, sys; sys.path.insert(0, "/content/repo")
import sweep
AMP = globals().get("AMP", "--amp")   # use "--no-amp" on a T4
os.makedirs(f"{RESULTS}/runs", exist_ok=True)
!rm -rf /content/repo/runs && ln -sfn {RESULTS}/runs /content/repo/runs
ALL = sweep.best_specs(f"{RESULTS}/tune*.jsonl")
CORE = [s for s in ALL if s.split(":")[0] in ("muon", "kaon", "sgd") or s.startswith("freon:0.6667")]   # cheap first
REST = [s for s in ALL if s not in CORE]
print("curve specs:", CORE, REST)
!python sweep.py scale --opts {" ".join(CORE + REST)} --rhos 1 {AMP} --seeds 42 --freon_impl {FREON_IMPL} --out {RESULTS}/curves.jsonl


In [ ]:
# Run after section 6 has finished (re-run it later to add the last curves). Writes the figure to Drive too.
!git -C /content/repo pull -q
!python /content/repo/plot_h1.py --runs {RESULTS}/runs/scale --out {RESULTS}/h1_curves.png
from IPython.display import Image; Image(f"{RESULTS}/h1_curves.png")